# Testing phase 1 functions

In [1]:
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
from PIL import Image
from phase1_setup import parse_weight, make_portion_id, is_blurry, validate_image, assign_splits
import cv2

# Test helper functions

## Test portions dict creation

In [2]:
dataset_dir = Path('/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets')
raw_dir = dataset_dir / 'raw'

In [3]:
food_type_dirs = sorted([d for d in raw_dir.iterdir() if d.is_dir()])
print(len(food_type_dirs))
print(food_type_dirs[1])

10
/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Gnocchi


In [ ]:

portions = []
skipped_folders = []
food_type_counters = {}


for food_type_dir in food_type_dirs:
    food_type = food_type_dir.name
    food_type_counters[food_type] = 0
    # print(food_type)
    portion_dirs = sorted([d for d in food_type_dir.iterdir() if d.is_dir()])
    # print(portion_dirs)
    
    for portion_dir in portion_dirs:
        weight = parse_weight(portion_dir.name)    

        if weight is None:
            print(f"Skipping {portion_dir} due to unparseable weight.")
            skipped_folders.append(portion_dir)
            continue
        
        food_type_counters[food_type] += 1
        portion_id = make_portion_id(food_type, food_type_counters[food_type])   
        portions.append({
            "portion_id": portion_id,
            "food_type": food_type,
            "weight": weight,
            "original_folder": portion_dir
        })

portion = portions[3]
print(portion)

{'portion_id': 'cananderli_004', 'food_type': 'Cananderli', 'weight': 250.0, 'originnal_folder': PosixPath('/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Cananderli/250g')}


## Test the dataset split

In [5]:
all_ids = [p['portion_id'] for p in portions]
all_types = [p['food_type'] for p in portions]
all_dirs = [p['originnal_folder'] for p in portions]
all_weights = [p['weight'] for p in portions]

print(all_ids[9:13])
print(all_types[9:13])

['cananderli_010', 'gnocchi_001', 'gnocchi_002', 'gnocchi_003']
['Cananderli', 'Gnocchi', 'Gnocchi', 'Gnocchi']


In [6]:
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42


x = assign_splits(all_ids, all_types)
print(x)


{'cananderli_003': 'train', 'cananderli_002': 'train', 'risotto_002': 'train', 'kiwi_003': 'train', 'penne_al_ragu_009': 'train', 'kiwi_006': 'train', 'risotto_003': 'train', 'kiwi_007': 'train', 'ravioli_008': 'train', 'pizza_006': 'train', 'cananderli_001': 'train', 'penne_al_ragu_003': 'train', 'kiwi_004': 'train', 'spaghetti_in_bianco_009': 'train', 'gnocchi_005': 'train', 'penne_al_ragu_006': 'train', 'gnocchi_009': 'train', 'kiwi_009': 'train', 'pecorino_grattuggiato_009': 'train', 'penne_al_ragu_004': 'train', 'ravioli_001': 'train', 'pizza_007': 'train', 'spaghetti_in_bianco_008': 'train', 'pecorino_grattuggiato_006': 'train', 'pizza_003': 'train', 'gnocchi_008': 'train', 'pizza_005': 'train', 'ravioli_007': 'train', 'pecorino_grattuggiato_005': 'train', 'spaghetti_in_bianco_003': 'train', 'cananderli_006': 'train', 'mela_001': 'train', 'mela_002': 'train', 'risotto_001': 'train', 'mela_008': 'train', 'ravioli_004': 'train', 'pecorino_grattuggiato_008': 'train', 'gnocchi_004': 

In [7]:
df = pd.DataFrame(
    {
        "portion_id": all_ids,
        "food_type": all_types,
        "dir": all_dirs,
        "weight": all_weights
    }
)

df["split"] = df["portion_id"].map(x)
df.head(10)

,portion_id,food_type,dir,weight,split
0,cananderli_001,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,100.0,train
1,cananderli_002,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,150.0,train
2,cananderli_003,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,200.0,train
3,cananderli_004,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,250.0,test
4,cananderli_005,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,25.0,val
5,cananderli_006,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,300.0,train
6,cananderli_007,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,350.0,val
7,cananderli_008,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,400.0,train
8,cananderli_009,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,50.0,train
9,cananderli_010,Cananderli,/home/akis/Documents/Projects/GitHubProjects/V...,75.0,train


In [14]:
for portion in portions:
    portion["split"] = x[portion["portion_id"]]

print(portions[3])

{'portion_id': 'cananderli_004', 'food_type': 'Cananderli', 'weight': 250.0, 'originnal_folder': PosixPath('/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Cananderli/250g'), 'split': 'test'}


In [ ]:
old = portions[0]["originnal_folder"]
new = old.parent / portions[0]["portion_id"]
/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw

In [21]:
print(old)
print(new)  

/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Cananderli/100g
/home/akis/Documents/Projects/GitHubProjects/Vippstar/datasets/raw/Cananderli/cananderli_001
